# JSpace steering demo — replace answer with 'Pairs'

This notebook demonstrates asking the model `What's the capital city of US?` and using a pre-fitted Jacobian lens to steer the model toward the token `Pairs`. `lens.steer` measures next-token controllability; `lens.steer_generate` keeps the same residual write active during HuggingFace `generate()`. Follow the requirements in `walkthrough.ipynb` and ensure you have a fitted lens available (see `walkthrough.md`).

In [1]:
import torch
import transformers
import jlens
from IPython.display import display
from jlens.vis import notebook_iframe
import json, gzip

/Users/cyb/Documents/GitHub/jspace-research/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Device and dtype selection (adapt if needed)
if torch.cuda.is_available():
    device = torch.device("cuda")
    torch_dtype = torch.bfloat16
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = torch.device("mps")
    torch_dtype = torch.float16
else:
    device = torch.device("cpu")
    torch_dtype = torch.float32
device, torch_dtype

(device(type='mps'), torch.float16)

In [3]:
# Configure these to match your fitted lens/model
MODEL_NAME = "Qwen/Qwen3.5-4B"
LENS_REPO = "neuronpedia/jacobian-lens"
LENS_REVISION = "qwen-n1000"
LENS_FILE = {
    "Qwen/Qwen3.5-4B": "qwen3.5-4b/jlens/Salesforce-wikitext/Qwen3.5-4B_jacobian_lens_n1000.pt",
    "Qwen/Qwen3.6-27B": "qwen3.6-27b/jlens/Salesforce-wikitext/Qwen3.6-27B_jacobian_lens_n1000.pt",
}[MODEL_NAME]
MODEL_NAME, LENS_REPO, LENS_FILE

('Qwen/Qwen3.5-4B',
 'neuronpedia/jacobian-lens',
 'qwen3.5-4b/jlens/Salesforce-wikitext/Qwen3.5-4B_jacobian_lens_n1000.pt')

In [4]:
# Load model and tokenizer and wrap with jlens' LensModel
hf_model = transformers.AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch_dtype, trust_remote_code=True).to(device)
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = jlens.from_hf(hf_model, tokenizer)
print('model device:', getattr(model, 'input_device', device))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 23237.14it/s]
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 426/426 [00:00<00:00, 4768.09it/s]


model device: mps:0


In [5]:
# Load the pre-fitted Jacobian lens (from Hub or local path)
lens = jlens.JacobianLens.from_pretrained(LENS_REPO, filename=LENS_FILE, revision=LENS_REVISION)
lens

Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 20068.44it/s]


JacobianLens(d_model=2560, n_prompts=1000, source_layers=[0..30] (31 layers))

In [6]:
# The user prompt we will ask the model
prompt = "What's the capital city of USA?"
print('prompt:', prompt)
# Tokenize for steering call (single example batch)
ids = torch.tensor([tokenizer.encode(prompt, add_special_tokens=False)], dtype=torch.long, device=getattr(model, 'input_device', device))
print('input ids:', ids.tolist())

prompt: What's the capital city of USA?
input ids: [[3710, 579, 279, 6511, 3177, 314, 7052, 30]]


In [7]:
# Choose the target token text you want to insert via JSpace steering
target_text = "Pairs"
tokens = tokenizer.encode(target_text, add_special_tokens=False)
if len(tokens) == 0:
    raise RuntimeError('tokenizer produced no tokens for target_text')
target_token_id = tokens[0]
print('target_text:', target_text, '-> token ids:', tokens)
print('using target_token_id:', target_token_id)

target_text: Pairs -> token ids: [52405]
using target_token_id: 52405


In [8]:
# Run a steering intervention at default strength and inspect the effect
strength = 0.2
result = lens.steer(model, ids, target_token_id=target_token_id, strength=strength)
print('clean target rank -> steered target rank:', result.clean_target_ranks.item(), '->', result.steered_target_ranks.item())

# Build a comparison visualization (if running in notebook with display)
comparison = jlens.compute_steering_comparison(model, lens, result, last_n_tokens=32, mask_display=True)
page = jlens.build_steering_comparison_page(comparison, title=f"Steering: {target_text}", description="Steering the next-token toward a chosen token (controllability demo).")
display(notebook_iframe(page, height=700))

clean target rank -> steered target rank: 98119 -> 0


In [9]:
# Decode and display clean and steered top tokens and top-5 candidates
clean_top_id = int(result.clean_top_token_ids[0])
steered_top_id = int(result.steered_top_token_ids[0])
clean_top_token = tokenizer.decode([clean_top_id])
steered_top_token = tokenizer.decode([steered_top_id])
print('clean top-1 token:', clean_top_token, '-> id', clean_top_id)
print('steered top-1 token:', steered_top_token, '-> id', steered_top_id)
# Show top-5 tokens from logits
clean_logits = result.clean_logits[0]
steered_logits = result.steered_logits[0]
def topk_tokens(logits, k=5):
    vals, ids = logits.topk(k)
    return [(int(i), tokenizer.decode([int(i)]), float(v)) for v,i in zip(vals.tolist(), ids.tolist())]
print('clean top-5:', topk_tokens(clean_logits,5))
print('steered top-5:', topk_tokens(steered_logits,5))
# Greedy one-step continuation using steered top token
next_token = steered_top_id
input_ids = result.input_ids.cpu()
generated = tokenizer.decode((input_ids[0].tolist() + [int(next_token)]))
print('greedy one-step generated sequence (steered):', generated)

clean top-1 token: 

 -> id 271
steered top-1 token: Pairs -> id 52405
clean top-5: [(271, '\n\n', 20.25), (198, '\n', 19.125), (471, ' -', 17.125), (3437, ' What', 16.5), (318, ' (', 16.125)]
steered top-5: [(52405, 'Pairs', 28.5), (30, '?', 20.375), (11, ',', 17.875), (13139, ' pairs', 17.625), (198, '\n', 16.75)]
greedy one-step generated sequence (steered): What's the capital city of USA?Pairs


In [11]:
# Generate continuations: clean vs inference-time J-space steering.
# Pass the same `ids` used by lens.steer so the generate prefix matches.
max_new_tokens = 128
generate_kwargs = dict(
    attention_mask=torch.ones_like(ids),
    max_new_tokens=max_new_tokens,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)

clean_gen_ids = hf_model.generate(ids, **generate_kwargs)
clean_generated = tokenizer.decode(clean_gen_ids[0], skip_special_tokens=True)
print('Clean generation:', clean_generated)

prompt_pass_ids = lens.steer_generate(
    hf_model,
    model,
    ids,
    target_token_id=target_token_id,
    strength=strength,
    decode_mode="prompt_pass",
    max_new_tokens=max_new_tokens,
)
prompt_pass_generated = tokenizer.decode(prompt_pass_ids[0], skip_special_tokens=True)
print('Steered generation (prompt_pass):', prompt_pass_generated)

every_step_ids = lens.steer_generate(
    hf_model,
    model,
    ids,
    target_token_id=target_token_id,
    strength=strength,
    decode_mode="every_step",
    max_new_tokens=max_new_tokens,
)
every_step_generated = tokenizer.decode(every_step_ids[0], skip_special_tokens=True)
print('Steered generation (every_step):', every_step_generated)


[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Clean generation: What's the capital city of USA?

<think>
Thinking Process:

1.  **Identify the core question:** The user is asking for the capital city of the USA (United States of America).

2.  **Retrieve knowledge:** Access general knowledge about the United States of America.
    *   Country: United States of America (USA)
    *   Capital City: Washington, D.C. (District of Columbia)

3.  **Formulate the answer:** State the capital city clearly.
    *   Draft: The capital city of the USA is Washington, D.C.

4.  **Review and refine:** Is there any ambiguity? No. Is it concise? Yes.
    *   Final Answer: Washington, D.C.

5.  **Output:** Washington, D.C.cw
</think>

The capital city of the USA is **Washington, D.C.** (District of Columbia).


[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Clean CoT generation: What's the capital city of USA? Let's think step by step.

<think>
Thinking Process:

1.  **Analyze the Request:**
    *   Question: "What's the capital city of USA?"
    *   Constraint: "Let's think step by step."
    *   Goal: Provide the correct answer while demonstrating a step-by-step reasoning process.

2.  **Identify the Core Fact:**
    *   The question asks for the capital city of the United States of America (USA).
    *   Common knowledge: Washington, D.C.

3.  **Formulate the Step-by-Step Reasoning:**
    *   Step 1: Identify the country in question (United States of America).
    *   Step 2: Recall or verify the definition of a capital city (the city where the government, specifically the executive, legislative, and judicial branches, are located).
    *   Step 3: Retrieve knowledge about the specific capital of the USA.
    *   Step 4: Confirm the name of the city (Washington, D.C.).
    *   Step 5: Distinguish it from other major cities (like New Yo

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Steered forced continuation: What's the capital city of USA?Pairs: 1. 1. 2. 2. 3. 3. 4. 4. 5. 5. 6. 6. 7. 7. 8. 8. 9. 9. 10. 10. 11. 11. 12. 12. 13. 13. 14. 14. 15. 15. 16. 16. 17. 17. 18. 18. 19. 19. 20. 20. 21. 21. 22. 22. 23. 23. 24. 24. 25. 25. 26. 26. 27. 27. 28. 28. 29. 29. 30. 30. 31. 31. 32. 32. 33. 33. 34. 34. 35. 35. 36. 36. 37. 37. 38. 38. 39. 39. 40. 40. 41. 41. 42. 42. 43. 43. 44. 44. 45. 45. 46. 46. 47. 47. 48. 48. 49. 49. 50. 50. 51. 51. 52. 52. 53. 53. 54. 54. 55. 55. 56. 56. 57. 57. 58. 58. 59. 59. 60. 60. 61. 61. 62. 62. 63. 63. 64. 64. 65. 65. 66. 66. 67. 67. 68. 68. 69. 69. 70. 70. 71. 71. 72. 72. 73. 73. 74. 74. 75. 75. 76. 76. 77. 77. 78. 78. 79. 79. 80. 80. 81. 81. 82. 82. 83. 83. 84. 84. 85. 85. 86. 86. 87. 87. 88. 88. 89. 89. 90. 90. 91. 91. 92. 92. 93. 93. 94. 94. 95. 95. 96. 96. 97. 97. 98. 98. 99. 99. 100. 100. 101. 101. 102. 102. 103. 103. 104. 104. 105. 105. 106. 106. 107. 107. 108. 108. 109. 109. 110. 110. 111. 111. 112. 112. 113. 113. 114. 114. 115. 115.

## Comparison: Before vs After Steering

The cell below prints the clean (before steering) and steered outputs so you can compare next-token diagnostics from `lens.steer` with full continuations from `lens.steer_generate` under `prompt_pass` vs `every_step`.

In [12]:
# Side-by-side comparison: clean vs steered
print('=== Top-1 tokens (lens.steer, next token) ===')
print('Clean :', clean_top_token, f'(id {clean_top_id})')
print('Steered:', steered_top_token, f'(id {steered_top_id})')

print('=== Top-5 tokens (clean) ===')
for i, tok, score in topk_tokens(clean_logits,5):
    print(f'{i}: {tok} (score={score:.4f})')

print('=== Top-5 tokens (steered) ===')
for i, tok, score in topk_tokens(steered_logits,5):
    print(f'{i}: {tok} (score={score:.4f})')

print('=== Greedy continuations (hooks during generate) ===')
print('Clean generation:', clean_generated)
print('Steered prompt_pass:', prompt_pass_generated)
print('Steered every_step:', every_step_generated)

=== Top-1 tokens ===
Clean : 

 (id 271)
Steered: Pairs (id 52405)
=== Top-5 tokens (clean) ===
271: 

 (score=20.2500)
198: 
 (score=19.1250)
471:  - (score=17.1250)
3437:  What (score=16.5000)
318:  ( (score=16.1250)
=== Top-5 tokens (steered) ===
52405: Pairs (score=28.5000)
30: ? (score=20.3750)
11: , (score=17.8750)
13139:  pairs (score=17.6250)
198: 
 (score=16.7500)
=== Greedy continuations ===
Clean generation: What's the capital city of USA?

<think>
Thinking Process:

1.  **Identify the core question:** The user is asking for the capital city of the USA (United States of America).

2.  **Retrieve knowledge:** Access general knowledge about the United States of America.
    *   Country: United States of America (USA)
    *   Capital City: Washington, D.C. (District of Columbia)

3.  **Formulate the answer:** State the capital city clearly.
    *   Draft: The capital city of the USA is Washington, D.C.

4.  **Review and refine:** Is there any ambiguity? No. Is it concise? Yes.


### Notes
- `lens.steer` performs a single-forward intervention using the normalized row of `W_U J_l`; we selected the first token id of the tokenized `Pairs` text as the steering target. `"Pairs"` is that vocab token, not `"Paris"`.
- `lens.steer_generate` keeps the same last-prompt-token delta active during HuggingFace `generate()`. `prompt_pass` writes it only at the last prompt token (later tokens see it via the KV cache). `every_step` also writes it at each new token.
- Qwen3.5-4B may still emit a `<think>` block under `prompt_pass` and drift away from `Pairs`. `every_step` keeps boosting that token's direction and is not guaranteed to produce a coherent sentence.
- If `Pairs` tokenizes to multiple tokens, steering only the first subtoken is a simple demonstration; for multi-token targets you can steer sequentially or adapt the method.
- Adjust `strength` to make the intervention weaker or stronger; large values may push the model outside the linear approximation.
- Ensure `LENS_FILE` matches your model exactly (same `d_model`, layer count).